# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Fraud-Ops War Room

### Scenario

You've joined the **Fraud Operations** team at **PayKart**, a card-payments company. Last month's numbers looked wrong and
the Head of Fraud Ops has called a review. You have 30 days of transactions with the automated detector's score and the
investigators' confirmed labels (`unit3_ppt7_card_transactions.csv`). Their requests:

1. *"I don't want a list of alerts. Show me the whole month first — is anything unusual going on?"*
2. *"The investigators keep saying 'night-time, far-from-home' transactions are risky. Show me, across ALL our views at
   once, what those transactions look like."*
3. *"Engineering wants to build the interactive filter tool on our full history: 50 million transactions. Will a slider
   still feel instant?"*
4. *"The detector threshold is 0.5 and the team is drowning in alerts. We can review at most **150 alerts a month**. Where
   should the threshold be?"*
5. *"Give the investigators a tool where they can poke at the flagged cases themselves."*
6. *"Someone says it was a fraud ring in one city. Prove it — or kill the idea."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer in your own
words. Interactive (plotly) figures support zoom, pan, hover, selection and clicking.

## Part 0 — Setup (given)

Run this cell as-is. It loads `tx` and adds `night` (hour < 6) and a readable `label` column.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
# If plotly figures come out blank: classic Jupyter -> pio.renderers.default = "notebook";
# Google Colab -> "colab"; VS Code / JupyterLab work out of the box.
import plotly.io as pio

tx = pd.read_csv("unit3_ppt7_card_transactions.csv", parse_dates=["timestamp"])
tx["night"] = tx.hour < 6
tx["label"] = np.where(tx.is_fraud == 1, "confirmed fraud", "legitimate")

print(f"{len(tx):,} transactions | {tx.is_fraud.sum()} confirmed fraud | days {tx.day.min()}-{tx.day.max()}")
tx.head()

## Task 1 — Overview First (Request #1)

**Real-world usage:** Fraud dashboards start from a whole-period overview before any alert list.

**Why this technique:** Shneiderman's mantra — **overview first**, then zoom and filter, then details (Slide 30); a
**reordered** matrix makes patterns pop out (Slide 39).

**TODO:**
1. Plot daily transaction counts with daily confirmed fraud on a second axis.
2. Build a city × day matrix of confirmed-fraud **counts**, reorder rows and columns by their totals, and show it as a
   heatmap with labelled axes.
3. Print the (city, day) cells with the most fraud.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7))
# TODO 1: daily transactions (bars) + daily confirmed fraud (line, second axis)
daily = tx.groupby("day").agg(transactions=("txn_id", "size"), fraud=("is_fraud", "sum"))
ax = axes[0]; ax2 = ax.twinx()
ax.bar(daily.index, daily.transactions, color="lightgrey", label="transactions")
ax2.plot(daily.index, daily.fraud, "o-", color="#D9534F", lw=2, label="confirmed fraud")
ax.set(xlabel="day of month", ylabel="transactions / day", title="Overview: daily volume (grey) vs confirmed fraud (red)")
ax2.set_ylabel("confirmed fraud / day", color="#D9534F"); ax2.spines["right"].set_visible(True)
ax.set_xticks(daily.index)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="upper left")

# TODO 2: city x day matrix of fraud counts, rows/columns reordered by their totals
heat = (tx[tx.is_fraud == 1].pivot_table(index="city", columns="day", values="txn_id", aggfunc="count", fill_value=0)
          .reindex(index=sorted(tx.city.unique()), columns=range(tx.day.min(), tx.day.max() + 1), fill_value=0))
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index, heat.sum(axis=0).sort_values(ascending=False).index]
im = axes[1].imshow(heat.values, aspect="auto", cmap="Reds")
axes[1].set_xticks(range(heat.shape[1])); axes[1].set_xticklabels(heat.columns)
axes[1].set_yticks(range(heat.shape[0])); axes[1].set_yticklabels(heat.index)
axes[1].set(xlabel="day (columns reordered by total fraud)", ylabel="city (rows reordered by total fraud)",
            title="Confirmed fraud count per city x day (reordered matrix)")
plt.colorbar(im, ax=axes[1], label="confirmed fraud count")

plt.tight_layout(); plt.show()

# TODO 3: the (city, day) cells with the most fraud
print("Top (city, day) cells by confirmed fraud:")
print(heat.stack().sort_values(ascending=False).head(5).rename("fraud_count").to_string())

**Reflection:** What did the overview show that an alert list sorted by detector score would *not* have shown? Name the
lecture concept (Slide 38 or 39) that describes how you found it.

_Your answer:_

The overview shows that the **daily volume is flat and normal** (roughly 150–220 transactions a day) while **confirmed fraud suddenly spikes on days 18–20 (13, 17 and 12 cases, against 0–7 on other days)**. The city × day matrix shows that nearly all of it is in a single cell block: **Pune, days 18, 19 and 20 (13, 17 and 10 cases)**. A list of alerts sorted by detector score would only show 260 separate high-score rows and would never reveal that they cluster in one city on three days. I found it through **overview first** (Shneiderman, Slide 30) and **pattern discovery by reordering the matrix** (Slide 39): sorting rows and columns by their totals pushed the hot cells to the top-left corner.


## Task 2 — Linked Views for "Night-Time, Far-From-Home" (Request #2)

**Real-world usage:** Investigators validate rules of thumb by brushing them across a coordinated dashboard.

**Why this technique:** **Cross-view brushing / linked highlighting** reveals how a selection in one view is distributed
in all the others (Slides 8–12).

**TODO:**
1. Define the brush: `night == True` **and** `distance_from_home_km > 100`.
2. Draw four coordinated views with the brushed items in red over all items in grey: (a) hour vs. distance scatter
   (log y), (b) merchant category counts, (c) city counts, (d) amount histogram (log10).
3. Print: number brushed, fraud rate inside the brush vs. overall, and the share of *all* fraud the brush captures.

In [ ]:
# TODO 1: the brush
brush = tx.night & (tx.distance_from_home_km > 100)

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
# TODO 2: four coordinated views, brushed items red over all items grey
# (a) hour vs distance (log y)
axes[0].scatter(tx.hour, tx.distance_from_home_km, s=6, c="lightgrey")
axes[0].scatter(tx.hour[brush], tx.distance_from_home_km[brush], s=10, c="#D9534F")
axes[0].set_yscale("log"); axes[0].set(xlabel="hour", ylabel="distance from home (km)", title="(a) hour vs distance")

# (b) merchant category counts
cat_all = tx.merchant_category.value_counts()
cat_b = tx[brush].merchant_category.value_counts().reindex(cat_all.index, fill_value=0)
axes[1].barh(cat_all.index, cat_all.values, color="lightgrey"); axes[1].barh(cat_b.index, cat_b.values, color="#D9534F")
axes[1].invert_yaxis(); axes[1].set(xlabel="transactions", title="(b) merchant category")

# (c) city counts
city_all = tx.city.value_counts()
city_b = tx[brush].city.value_counts().reindex(city_all.index, fill_value=0)
axes[2].barh(city_all.index, city_all.values, color="lightgrey"); axes[2].barh(city_b.index, city_b.values, color="#D9534F")
axes[2].invert_yaxis(); axes[2].set(xlabel="transactions", title="(c) city")

# (d) amount histogram (log10)
bins = np.linspace(np.log10(tx.amount_inr.min()), np.log10(tx.amount_inr.max()), 30)
axes[3].hist(np.log10(tx.amount_inr), bins=bins, color="lightgrey", label="all")
axes[3].hist(np.log10(tx.amount_inr[brush]), bins=bins, color="#D9534F", label="brushed"); axes[3].legend()
axes[3].set(xlabel="log10(amount in INR)", ylabel="transactions", title="(d) amount")

plt.tight_layout(); plt.show()

# TODO 3: numbers
rate_in, rate_all = tx.is_fraud[brush].mean(), tx.is_fraud.mean()
share = tx.is_fraud[brush].sum() / tx.is_fraud.sum()
print(f"Brushed transactions: {brush.sum()}")
print(f"Fraud rate inside brush: {rate_in:.1%} vs overall {rate_all:.1%}  ({rate_in / rate_all:.0f}x higher)")
print(f"Share of ALL fraud captured by the brush: {share:.1%} ({tx.is_fraud[brush].sum()} of {tx.is_fraud.sum()})")

**Reflection:** The brush has a very high fraud *rate* — but what share of all fraud does it *capture*? What does that
tell the investigators about relying on this rule alone, and which linked view hinted at where the rest of the fraud is?

_Your answer:_

The brush (night-time and more than 100 km from home) contains 61 transactions and has a fraud rate of about **87 % against 2.2 % overall (about 40× higher)**, but it captures only **53 of the 120 confirmed frauds, i.e. 44 %**. So more than half of the fraud (67 cases) happens outside the rule, and an investigator who relies on this rule alone would miss most of it even though every hit looks very convincing. High precision does not mean high recall. The **city view (c)** hinted at where the rest is: Pune is far over-represented among the brushed items (25 of 61, although it is only ~13 % of all transactions), which points to the Pune cluster found in Task 1, and the category view (b) also showed concentration in Electronics and Gift Cards.


## Task 3 — Will the Slider Still Feel Instant? (Request #3)

**Real-world usage:** Interactive tools are sized against latency budgets before they're built.

**Why this technique:** **Dynamic queries** need feedback within ~0.1 s (Slides 15–16).

**TODO:**
1. Build a 1-million-row table by repeating `tx`, and measure the time of the filter
   `(detector_score >= 0.6) & (amount_inr > 3000)` (take the best of 5 runs).
2. Compute the per-item cost (µs) and the maximum number of rows filterable within 0.1 s on this machine.
3. Estimate the response time for 50,000,000 rows and the % of rows that would need to be skipped (pre-indexed /
   pre-aggregated) to meet the 0.1 s budget.

In [ ]:
big = pd.concat([tx] * 182, ignore_index=True)

# TODO 1: best of 5 runs of the filter
times = []
for _ in range(5):
    t0 = time.perf_counter()
    _ = big[(big.detector_score >= 0.6) & (big.amount_inr > 3000)]
    times.append(time.perf_counter() - t0)
best_s = min(times)

# TODO 2: per-item cost and max rows within the 0.1 s budget
per_item_us = best_s / len(big) * 1e6
max_rows = 0.1 / (best_s / len(big))

# TODO 3: extrapolate to 50 M rows
t_50m = best_s / len(big) * 50_000_000
skip_pct = max(0.0, (1 - 0.1 / t_50m) * 100)

print(f"{len(big):,} rows in {best_s*1000:.1f} ms -> {per_item_us:.4f} us/row -> max {max_rows:,.0f} rows in 0.1 s")
print(f"50M rows -> {t_50m:.2f} s; must skip {skip_pct:.1f}% of rows to stay under 0.1 s")

**Reflection:** Besides indexing, name one **reduce** technique from the lecture (Slide 5 / 14) that would let the slider
stay instant on 50M rows, and what the analyst would lose by using it.

_Your answer:_

One **reduce** technique is **aggregation** (pre-binning into a data cube, for example counts and fraud rates per hour × city × category, so the slider filters a few thousand bins instead of 50 million rows); another is **sampling**. The analyst would lose **individual records and fine detail**: a binned view cannot show a single transaction, and with fraud at only ~2 % random sampling could drop the rare fraud cases entirely, so the aggregated bins hide outliers and the exact row must be fetched on demand (details-on-demand) from the full data.


## Task 4 — Steer the Threshold to the Team's Capacity (Request #4)

**Real-world usage:** Alert thresholds are set against investigator capacity, not just statistical optimality.

**Why this technique:** **Human-in-the-loop** steering of a model parameter (Slide 37) evaluated with **precision, recall,
F1** (Slides 40–41).

**TODO:**
1. For thresholds 0.05, 0.10, …, 0.95 compute alerts (flagged count), TP, FP, FN, precision, recall and F1.
2. Find the threshold with the **highest recall** among those producing **≤ 150 alerts**. Also find the best-F1 threshold.
3. Plot the precision–recall curve and mark the current threshold (0.5), the capacity-constrained choice and the best-F1
   choice.
4. Print a one-line recommendation.

In [ ]:
rows = []
# TODO 1: sweep the threshold
for th in np.round(np.arange(0.05, 0.96, 0.05), 2):
    flagged_ = tx.detector_score >= th
    tp = int((flagged_ & (tx.is_fraud == 1)).sum()); fp = int((flagged_ & (tx.is_fraud == 0)).sum()); fn = int((~flagged_ & (tx.is_fraud == 1)).sum())
    p_ = tp / (tp + fp) if tp + fp else 0.0; r_ = tp / (tp + fn) if tp + fn else 0.0
    f_ = 2 * p_ * r_ / (p_ + r_) if p_ + r_ else 0.0
    rows.append(dict(threshold=th, alerts=tp + fp, TP=tp, FP=fp, FN=fn, precision=p_, recall=r_, F1=f_))

pr = pd.DataFrame(rows)
try:
    from IPython.display import display
except ImportError:
    display = print
display(pr.round(3))

# TODO 2: capacity-constrained choice (<= 150 alerts, max recall) and best-F1 choice
capacity_choice = pr[pr.alerts <= 150].sort_values(["recall", "precision"], ascending=False).iloc[0]
best_f1 = pr.sort_values("F1", ascending=False).iloc[0]
current = pr[pr.threshold == 0.5].iloc[0]

# TODO 3: precision-recall curve with the three choices marked
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pr.recall, pr.precision, "o-", color="#4C72B0", alpha=0.6)
for row, lab, c in [(current, "current 0.5", "grey"), (capacity_choice, "capacity (<=150 alerts)", "#D9534F"), (best_f1, "best F1", "green")]:
    ax.plot(row.recall, row.precision, "o", ms=13, color=c, label=f"{lab}: threshold {row.threshold:.2f}, {int(row.alerts)} alerts")
ax.set(xlabel="recall", ylabel="precision", title="Precision-recall trade-off as the threshold moves"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# TODO 4: one-line recommendation
print(f"Recommendation: move the threshold from 0.50 ({int(current.alerts)} alerts, recall {current.recall:.0%}) to {capacity_choice.threshold:.2f} "
      f"-> {int(capacity_choice.alerts)} alerts, precision {capacity_choice.precision:.0%}, recall {capacity_choice.recall:.0%} (best F1 would be at {best_f1.threshold:.2f}, {int(best_f1.alerts)} alerts).")

**Reflection:** The capacity-constrained threshold and the best-F1 threshold may differ. Who should make that call — the
model or the Head of Fraud Ops — and why does visual analytics insist on keeping a human in the loop here (Slides 35–37)?

_Your answer:_

The **Head of Fraud Ops should make the call**, not the model. The model can only optimise a number such as F1 (the best-F1 threshold here is 0.85, with 128 alerts), but it knows nothing about the real investigator capacity (150 alerts a month), the cost of missing a fraud versus wasting an investigator's time, or business and customer-impact policies. Visual analytics keeps the human in the loop (Slides 35–37) because the human sees the precision–recall trade-off, the alert counts and the context, and can steer the parameter. Here the capacity-constrained choice (0.80: 144 alerts, precision ≈ 58 %, recall ≈ 70 %) is almost as good as the best F1 (0.636 vs 0.637), so the human can pick the one that fits the team.


## Task 5 — An Interactive Tool for the Investigators (Request #5)

**Real-world usage:** Investigators triage flagged cases themselves with zoom, filter and hover-for-details.

**Why this technique:** **Details-on-demand / tooltips** (Slides 22–23), **dynamic queries** (Slide 15) and **drill-down**
(Slide 24) — Shneiderman's mantra as an actual tool.

**TODO:**
1. Interactive scatter (plotly) of all transactions with `detector_score >= 0.3`: x = timestamp, y = amount (log),
   colour = `label`, with a tooltip showing txn_id, customer, city, category, distance and score. Set `dragmode="select"`.
2. Interactive **sunburst** for drill-down: city → merchant category, sized by transaction count, coloured by fraud rate.

In [ ]:
# TODO 1: interactive scatter of all transactions with detector_score >= 0.3
flagged = tx[tx.detector_score >= 0.3]
fig = px.scatter(flagged, x="timestamp", y="amount_inr", color="label", log_y=True,
                 color_discrete_map={"legitimate": "#4C72B0", "confirmed fraud": "#D9534F"},
                 hover_name="txn_id",
                 hover_data={"customer_id": True, "city": True, "merchant_category": True, "distance_from_home_km": ":.0f",
                             "detector_score": ":.3f", "label": False, "timestamp": "|%d %b %H:%M"},
                 title="Flagged cases (score >= 0.3): hover for details, drag to select", height=480)
fig.update_traces(marker_size=7)
fig.update_layout(dragmode="select")
fig.show()

# TODO 2: sunburst drill-down, city -> merchant category
agg = tx.groupby(["city", "merchant_category"]).agg(transactions=("txn_id", "size"), fraud=("is_fraud", "sum")).reset_index()
agg["fraud_rate_pct"] = agg.fraud / agg.transactions * 100
fig2 = px.sunburst(agg, path=[px.Constant("All cities"), "city", "merchant_category"], values="transactions",
                   color="fraud_rate_pct", color_continuous_scale="Reds", range_color=[0, 25],
                   hover_data={"fraud": True}, title="Drill-down: click a city to zoom in, click the centre to roll up", height=560)
fig2.show()

**Reflection:** Tooltips are great for one case at a time. Why would they be a *frustrating* way for an investigator to get
an overview of the 400 flagged cases (Slide 23), and which view from this notebook fixes that?

_Your answer:_

Tooltips show **one record at a time**, so to understand 400 flagged cases the investigator would have to hover over each point and remember what they saw, which is slow, has a high interaction cost and overloads working memory; they also hide the patterns, because nothing in a tooltip shows how the cases are distributed. An **overview** fixes that: the **city × day heatmap (Task 1)** or the **sunburst drill-down (Task 5)**, which shows fraud rate by city and category at a glance, and the linked views of Task 2 that show how a selection is distributed across the other views. Tooltips should then be used only for details on demand once the overview has pointed at a region.


## Task 6 — Prove or Kill the "Fraud Ring" Hypothesis (Request #6)

**Real-world usage:** Before escalating to law enforcement, the team must rule out simpler explanations.

**Why this technique:** **Hypothesis testing** with visual + statistical confirmation and an explicit **counter-example
search** to guard against confirmation bias (Slides 42–43).

**TODO:**
1. State the hypothesis precisely using what you found in Task 1 (city and day range).
2. Compute the fraud rate inside vs. outside that city–day window and a chi-square p-value
   (`scipy.stats.chi2_contingency` on `pd.crosstab(in_window, tx.is_fraud)`).
3. Counter-example searches: fraud rate for (a) the same city on other days, (b) other cities on the same days.
4. Visual confirmation: a bar chart of the four groups (window / same city other days / other cities same days / rest).
5. Print your verdict.

In [ ]:
from scipy.stats import chi2_contingency

# TODO 1: hypothesis -- "a fraud ring in Pune on days 18-20"
RING_CITY, RING_DAYS = "Pune", (18, 20)
in_city = tx.city == RING_CITY
in_days = tx.day.between(*RING_DAYS)
in_window = in_city & in_days
print(f"H1: fraud rate is much higher for {RING_CITY} on days {RING_DAYS[0]}-{RING_DAYS[1]} than elsewhere.")

# TODO 2: inside vs outside, chi-square
table = pd.crosstab(in_window, tx.is_fraud)
chi2, pval, _, _ = chi2_contingency(table)
r_in, r_out = tx.is_fraud[in_window].mean(), tx.is_fraud[~in_window].mean()
print(f"Inside window: {tx.is_fraud[in_window].sum()}/{in_window.sum()} = {r_in:.1%} | outside: {r_out:.1%} | chi2 = {chi2:.1f}, p = {pval:.2e}")

# TODO 3: counter-example searches
same_city_other_days = in_city & ~in_days
other_cities_same_days = ~in_city & in_days
rest = ~in_city & ~in_days
r_a, r_b, r_rest = tx.is_fraud[same_city_other_days].mean(), tx.is_fraud[other_cities_same_days].mean(), tx.is_fraud[rest].mean()
print(f"(a) {RING_CITY}, other days:        {r_a:.1%}  (n={same_city_other_days.sum()})")
print(f"(b) other cities, days {RING_DAYS[0]}-{RING_DAYS[1]}: {r_b:.1%}  (n={other_cities_same_days.sum()})")
print(f"(c) rest of the data:           {r_rest:.1%}  (n={rest.sum()})")

# TODO 4: visual confirmation
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{RING_CITY}\ndays {RING_DAYS[0]}-{RING_DAYS[1]}\n(window)", f"{RING_CITY}\nother days", f"other cities\nsame days", "rest"]
vals = [r_in * 100, r_a * 100, r_b * 100, r_rest * 100]
bars = ax.bar(labels, vals, color=["#D9534F", "#4C72B0", "#4C72B0", "#4C72B0"])
for b_, v_ in zip(bars, vals): ax.text(b_.get_x() + b_.get_width() / 2, v_ + 0.3, f"{v_:.1f}%", ha="center")
ax.set(ylabel="fraud rate (%)", title="Fraud rate: is the spike specific to this city AND these days?")
plt.tight_layout(); plt.show()

# TODO 5: verdict
confirmed = pval < 0.001 and r_in > 5 * max(r_a, r_b, r_rest)
print("VERDICT:", "hypothesis SURVIVES both counter-example searches -> a localised burst (Pune, days 18-20), escalate for investigation."
      if confirmed else "hypothesis NOT supported -> look for another explanation.")

**Reflection:** Which of your two counter-example searches would have *killed* the hypothesis if it had come out
differently, and what alternative explanation would it have supported instead?

_Your answer:_

**Both** counter-example searches could have killed the idea. **(b) Other cities on the same days**: if fraud had also been high in other cities on days 18–20, the spike would be driven by the *dates* (a nationwide campaign, a data breach, or a detector or labelling change on those days), not by a Pune ring. In fact it was only 0.4 %. **(a) Pune on other days**: if Pune had a high rate on every day, the cause would be a *permanent* Pune factor (a compromised terminal or merchant, or a high-risk area), not a burst. In fact it was 1.9 %, near the 1.5 % baseline. Since neither alternative held, the hypothesis survives (37.4 % inside the window against 1.5 % outside, chi-square p ≈ 1e-136). Caveat: I chose the window after seeing the data, so the p-value is optimistic, and the data show a localised burst, not proof of a coordinated "ring" (the 40 cases involve 40 different customers).


## Task 7 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence note to the Head of Fraud Ops covering the whole notebook. It should:
- State what happened last month in one sentence, with numbers.
- Recommend the alert threshold and justify it with capacity, precision/recall and the human-in-the-loop idea.
- Recommend the **one** interactive view to build first for the investigators, and justify it with at least one lecture
  idea (overview first, linked brushing, dynamic queries with the 0.1 s budget, details-on-demand).
- Name the feedback loop (Slide 51) that should be added so next month's detector is better.

_Your final note here:_

Last month PayKart had 5,520 transactions with 120 confirmed frauds (2.2 %), but 40 of them (a third) were a single burst in **Pune on days 18–20**, where the fraud rate was 37 % against 1.5 % elsewhere and neither Pune on other days (1.9 %) nor other cities on the same days (0.4 %) showed it, so this is a localised event worth escalating. I recommend moving the threshold from 0.50 (260 alerts, which is more than the 150-alert capacity) to **0.80**: 144 alerts, 58 % precision and 70 % recall, almost the same F1 as the best (0.85), with the final choice staying with the Head of Fraud Ops as a human-in-the-loop decision. The first interactive view to build is a **city × day overview heatmap linked (brushing) to the flagged-case scatter with details-on-demand**, because overview first would have shown the Pune burst immediately and the brush shows what the selected cases look like in every other view, while on 50M rows it must be pre-aggregated to stay within the 0.1 s dynamic-query budget. Finally, add a **feedback loop (Slide 51)**: investigators' confirm or reject decisions on the alerts are fed back as labels to retrain the detector, so next month's scores (and the Pune pattern) are better.
